In [46]:
import numpy as np
import pandas as pd
from sklearn.metrics import auc, average_precision_score, precision_recall_curve, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
import tensorflow as tf
from tensorflow.keras import Input, layers, models
from tensorflow.keras.layers import Input, Dense, Concatenate, Dropout , BatchNormalization, Activation
from tensorflow.keras.metrics import AUC, Precision, Recall
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model, Sequential






print("import xong ")

import xong 


In [8]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/nambo2004/labels/labels_11000_dong.csv
/kaggle/input/datasets/nambo2004/benh-vien-b-private/private_B_10000_dong.csv
/kaggle/input/datasets/nambo2004/share-feature/shared_data_11000_dong.csv
/kaggle/input/datasets/nambo2004/benh-vien-a-private/private_A_1000_dong.csv


In [14]:
# 1. Bảng nhãn (labels)
df_labels = pd.read_csv('/kaggle/input/datasets/nambo2004/labels/labels_11000_dong.csv')
print("=== BẢNG LABELS ===")
display(df_labels.head(20))
print("Số Column - Số ROW ")
print(df_labels.shape)

# 2. Bảng bệnh viện B (private B)
df_b = pd.read_csv('/kaggle/input/datasets/nambo2004/benh-vien-b-private/private_B_10000_dong.csv')
print("=== BẢNG BỆNH VIỆN private B ===")
display(df_b.head(20))
print("Số Column - Số ROW ")
print(df_b.shape)

# 3. Bảng dữ liệu dùng chung (shared data)
df_shared = pd.read_csv('/kaggle/input/datasets/nambo2004/share-feature/shared_data_11000_dong.csv')
print("=== BẢNG SHARED DATA ===")
display(df_shared.head(20))
print("Số Column - Số ROW ")
print(df_shared.shape)

# 4. Bảng bệnh viện A (private A)
df_a = pd.read_csv('/kaggle/input/datasets/nambo2004/benh-vien-a-private/private_A_1000_dong.csv')
print("=== BẢNG BỆNH VIỆN private A ===")
display(df_a.head(20))
print("Số Column - Số ROW ")
print(df_a.shape)

=== BẢNG LABELS ===


,hadm_id,mortality
0,22600049,0
1,26774271,0
2,23181353,0
3,21453672,0
4,29740189,0
5,28991043,0
6,21610632,0
7,25929738,1
8,23795034,0
9,24401840,1


Số Column - Số ROW 
(11000, 2)
=== BẢNG BỆNH VIỆN private B ===


,hadm_id,creatinine_max,bun_max,anion_gap,lactate_max,ph_min,potassium_mean,sodium_mean,chloride_mean,wbc_max,hemoglobin_min,platelets_min,inr_max,glucose_mean,bilirubin_max
0,27021321,1.7,64.0,18.0,1.4,7.33,3.700000,147.333333,106.666667,12.5,8.5,184.0,2.5,134.000000,0.3
1,20772846,0.8,13.0,17.0,1.2,7.34,4.450000,139.000000,105.000000,22.6,14.9,368.0,1.1,93.000000,0.4
2,21905572,1.6,13.0,15.0,3.2,7.23,3.800000,146.500000,121.000000,22.5,9.4,56.0,1.6,90.500000,0.2
3,28855134,3.4,27.0,17.0,1.1,7.13,3.666667,142.666667,114.000000,4.4,9.8,123.0,1.1,101.666667,0.7
4,28959049,0.8,14.0,12.0,1.5,7.31,4.450000,142.000000,102.500000,10.5,10.1,337.0,1.3,87.000000,0.8
5,22580364,2.0,50.0,21.0,1.7,7.38,4.050000,132.500000,94.500000,11.9,14.6,189.0,1.1,309.000000,0.5
6,22030954,1.1,20.0,28.0,8.6,7.27,4.400000,145.000000,103.000000,20.8,13.4,219.0,1.3,359.000000,0.5
7,28251507,4.2,99.0,17.0,1.3,7.26,4.625000,141.750000,102.500000,36.7,10.0,289.0,1.2,326.000000,0.6
8,21109754,1.6,27.0,12.0,1.8,7.21,4.300000,140.000000,110.000000,11.9,12.8,124.0,1.2,133.000000,0.4
9,23601441,0.9,23.0,11.0,1.2,7.36,4.200000,132.000000,100.000000,9.0,4.5,253.0,1.1,86.500000,0.3


Số Column - Số ROW 
(10000, 15)
=== BẢNG SHARED DATA ===


,hadm_id,gender,age,cardiovascular,neurological,pulmonary,diabetes,renal,liver,cancer,mental_substance,hem_metabolic,autoimmune,gcs_min,weight_kg
0,22600049,0,61,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,80.2
1,26774271,0,67,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,55.5
2,23181353,0,79,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,80.2
3,21453672,1,70,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,80.2
4,29740189,0,67,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,80.2
5,28991043,0,62,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,80.2
6,21610632,1,63,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,104.0
7,25929738,0,70,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,80.2
8,23795034,1,62,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,80.2
9,24401840,0,89,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,80.2


Số Column - Số ROW 
(11000, 15)
=== BẢNG BỆNH VIỆN private A ===


,hadm_id,heart_rate_mean,sbp_mean,dbp_mean,mbp_mean,resp_rate_mean,temp_c_mean,spo2_mean,fio2_max,urine_output_24h
0,22600049,63.241379,121.800000,55.933333,73.466667,17.666667,36.650794,99.516129,50.0,3000.0
1,26774271,78.769231,107.814815,50.111111,68.222222,14.692308,36.444444,98.846154,100.0,3615.0
2,23181353,68.291667,123.212121,65.939394,90.666667,16.041667,37.629630,98.875000,40.0,1098.0
3,21453672,68.576923,101.741935,52.580645,68.129032,17.846154,36.896825,99.916667,100.0,3905.0
4,29740189,70.480000,110.923077,53.384615,70.923077,13.760000,36.777778,96.961538,100.0,4025.0
5,28991043,69.157895,111.500000,53.361111,67.361111,22.552632,37.194444,99.342105,40.0,1005.0
6,21610632,97.307692,109.533333,56.400000,71.033333,18.103448,36.694444,97.583333,100.0,1614.0
7,25929738,97.875000,108.090909,59.818182,73.818182,17.739130,37.277778,96.541667,50.0,850.0
8,23795034,80.285714,107.000000,60.448276,76.310345,14.428571,36.833333,97.178571,100.0,2060.0
9,24401840,76.772727,151.111111,43.611111,80.277778,22.363636,37.000000,100.000000,100.0,2015.0


Số Column - Số ROW 
(1000, 10)


In [17]:
# 1. Ghép dữ liệu cho bvA (1.000 dòng)
data_A = df_a.merge(df_shared, on='hadm_id').merge(df_labels, on='hadm_id')
print ("Benh vien A sau khi ghep them lable ")
display(data_A.head())
print(f"Gốc A: {len(df_a)} dòng  -->  Sau khi ghép data_A: {len(data_A)} dòng")



# 2. Ghép dữ liệu cho bvB (10.000 dòng)
data_B = df_b.merge(df_shared, on='hadm_id').merge(df_labels, on='hadm_id')
print ("Benh vien B sau khi ghep them lable ")
display(data_A.head())
print(f"Gốc B: {len(df_b)} dòng  -->  Sau khi ghép data_B: {len(data_B)} dòng")


Benh vien A sau khi ghep them lable 


,hadm_id,heart_rate_mean,sbp_mean,dbp_mean,mbp_mean,resp_rate_mean,temp_c_mean,spo2_mean,fio2_max,urine_output_24h,...,diabetes,renal,liver,cancer,mental_substance,hem_metabolic,autoimmune,gcs_min,weight_kg,mortality
0,22600049,63.241379,121.800000,55.933333,73.466667,17.666667,36.650794,99.516129,50.0,3000.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,80.2,0
1,26774271,78.769231,107.814815,50.111111,68.222222,14.692308,36.444444,98.846154,100.0,3615.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,55.5,0
2,23181353,68.291667,123.212121,65.939394,90.666667,16.041667,37.629630,98.875000,40.0,1098.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,80.2,0
3,21453672,68.576923,101.741935,52.580645,68.129032,17.846154,36.896825,99.916667,100.0,3905.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,80.2,0
4,29740189,70.480000,110.923077,53.384615,70.923077,13.760000,36.777778,96.961538,100.0,4025.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,80.2,0


Gốc A: 1000 dòng  -->  Sau khi ghép data_A: 1000 dòng
Benh vien B sau khi ghep them lable 


,hadm_id,heart_rate_mean,sbp_mean,dbp_mean,mbp_mean,resp_rate_mean,temp_c_mean,spo2_mean,fio2_max,urine_output_24h,...,diabetes,renal,liver,cancer,mental_substance,hem_metabolic,autoimmune,gcs_min,weight_kg,mortality
0,22600049,63.241379,121.800000,55.933333,73.466667,17.666667,36.650794,99.516129,50.0,3000.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,80.2,0
1,26774271,78.769231,107.814815,50.111111,68.222222,14.692308,36.444444,98.846154,100.0,3615.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,55.5,0
2,23181353,68.291667,123.212121,65.939394,90.666667,16.041667,37.629630,98.875000,40.0,1098.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,80.2,0
3,21453672,68.576923,101.741935,52.580645,68.129032,17.846154,36.896825,99.916667,100.0,3905.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,80.2,0
4,29740189,70.480000,110.923077,53.384615,70.923077,13.760000,36.777778,96.961538,100.0,4025.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,80.2,0


Gốc B: 10000 dòng  -->  Sau khi ghép data_B: 10000 dòng


In [33]:
# ==========================================
# 1. CHIA TẬP TRAIN / TEST (80/20) VÀ GIỮ TỶ LỆ TỬ VONG ĐỀU (STRATIFY)
# ==========================================
train_A, test_A = train_test_split(data_A, test_size=0.2, random_state=42, stratify=data_A['mortality'])
train_B, test_B = train_test_split(data_B, test_size=0.2, random_state=42, stratify=data_B['mortality'])
print(f"Train A: {len(train_A)} | Test A: {len(test_A)}")
print(f"Train B: {len(train_B)} | Test B: {len(test_B)}")

Train A: 800 | Test A: 200
Train B: 8000 | Test B: 2000


In [34]:

# 2. TÁCH DANH SÁCH CỘT ĐẶC TRƯNG
# ==========================================
# Lấy tên cột tự động từ các file ban đầu (nhớ trừ đi cột ID và Nhãn)
common_cols = [c for c in df_shared.columns if c not in ['hadm_id', 'mortality', 'mortality_1yr']]
spec_A_cols = [c for c in df_a.columns if c not in ['hadm_id', 'mortality', 'mortality_1yr']]
spec_B_cols = [c for c in df_b.columns if c not in ['hadm_id', 'mortality', 'mortality_1yr']]
print(f"Số biến chung: {len(common_cols)} | Riêng A: {len(spec_A_cols)} | Riêng B: {len(spec_B_cols)}")


Số biến chung: 14 | Riêng A: 9 | Riêng B: 14


In [35]:
# ==========================================
# 3. CHUẨN HÓA DỮ LIỆU (STANDARD SCALER)
# ==========================================
# --- BỆNH VIỆN A ---
scaler_common_A = StandardScaler()
X_common_train_A = scaler_common_A.fit_transform(train_A[common_cols])
X_common_test_A  = scaler_common_A.transform(test_A[common_cols])
scaler_spec_A = StandardScaler()
X_spec_train_A = scaler_spec_A.fit_transform(train_A[spec_A_cols])
X_spec_test_A  = scaler_spec_A.transform(test_A[spec_A_cols])
y_train_A = train_A['mortality'].values
y_test_A  = test_A['mortality'].values
# --- BỆNH VIỆN B ---
scaler_common_B = StandardScaler()
X_common_train_B = scaler_common_B.fit_transform(train_B[common_cols])
X_common_test_B  = scaler_common_B.transform(test_B[common_cols])
scaler_spec_B = StandardScaler()
X_spec_train_B = scaler_spec_B.fit_transform(train_B[spec_B_cols])
X_spec_test_B  = scaler_spec_B.transform(test_B[spec_B_cols])
y_train_B = train_B['mortality'].values
y_test_B  = test_B['mortality'].values
print("Chuẩn hóa hoàn tất! Dữ liệu đã sẵn sàng để train Neural Network.")

Chuẩn hóa hoàn tất! Dữ liệu đã sẵn sàng để train Neural Network.


In [36]:
# ==========================================
# 1. OVERSAMPLING BỆNH VIỆN A LÊN BẰNG B (GIỮ TỶ LỆ NHÃN)
# ==========================================
# Gộp tạm X và y của A lại để resample chung
train_A_temp = pd.DataFrame(X_common_train_A, columns=common_cols)
spec_df = pd.DataFrame(X_spec_train_A, columns=spec_A_cols)
train_A_temp = pd.concat([train_A_temp, spec_df], axis=1)
train_A_temp['target'] = y_train_A
target_size = len(X_common_train_B) # Bằng đúng số dòng train của B (khoảng 8.000)
oversampled_chunks = []
for label in [0, 1]:
    class_subset = train_A_temp[train_A_temp['target'] == label]
    class_fraction = len(class_subset) / len(train_A_temp)
    n_samples_needed = int(target_size * class_fraction)
    
    resampled_chunk = resample(
        class_subset, replace=True, n_samples=n_samples_needed, random_state=42
    )
    oversampled_chunks.append(resampled_chunk)
df_A_os = pd.concat(oversampled_chunks).sample(frac=1, random_state=42)
# Tách lại ra X_common, X_spec, và y sau khi đã oversample
X_common_train_A_os = df_A_os[common_cols].values
X_spec_train_A_os = df_A_os[spec_A_cols].values
y_train_encoded_A_os = to_categorical(df_A_os['target'].values)
y_train_encoded_B = to_categorical(y_train_B)
# Bù đắp vài dòng nếu làm tròn bị lẻ (đảm bảo A dài đúng bằng B)
diff = len(X_common_train_B) - len(X_common_train_A_os)
if diff > 0:
    X_common_train_A_os = np.vstack([X_common_train_A_os, X_common_train_A_os[:diff]])
    X_spec_train_A_os = np.vstack([X_spec_train_A_os, X_spec_train_A_os[:diff]])
    y_train_encoded_A_os = np.vstack([y_train_encoded_A_os, y_train_encoded_A_os[:diff]])
print(f"Số lượng dữ liệu sau khi Oversample: A={len(X_common_train_A_os)} | B={len(X_common_train_B)}")

Số lượng dữ liệu sau khi Oversample: A=8000 | B=8000


In [47]:
# 2. XÂY DỰNG MÔ HÌNH SHARED - PRIVATE 

# --- SHARED ENCODER (DÙNG CHUNG) ---
input_common_A = Input(shape=(len(common_cols),), name="X_common_A")
input_common_B = Input(shape=(len(common_cols),), name="X_common_B")
shared_layers = Sequential([
    Dense(256), BatchNormalization(), Activation('relu'), Dropout(0.3),
    Dense(128), BatchNormalization(), Activation('relu'),
    Dense(64, activation='relu')
], name="Shared_Encoder")

out_shared_A = shared_layers(input_common_A)
out_shared_B = shared_layers(input_common_B)

# --- PRIVATE ENCODERS (DÙNG RIÊNG) ---
input_spec_A = Input(shape=(len(spec_A_cols),), name="X_spec_A")
private_A_layers = Sequential([
    Dense(128, activation='relu'), BatchNormalization(), Dense(64, activation='relu')
], name="Private_Encoder_A")
out_private_A = private_A_layers(input_spec_A)


input_spec_B = Input(shape=(len(spec_B_cols),), name="X_spec_B")
private_B_layers = Sequential([
    Dense(128, activation='relu'), BatchNormalization(), Dense(64, activation='relu')
], name="Private_Encoder_B")
out_private_B = private_B_layers(input_spec_B)
# --- GỘP & DỰ ĐOÁN (CLASSIFIER HEAD) ---
def classifier_head(h_shared, h_private, name):
    concat = Concatenate()([h_shared, h_private])
    x = Dense(64, activation='relu')(concat)
    x = Dropout(0.2)(x)
    return Dense(2, activation='softmax', name=name)(x)
output_A = classifier_head(out_shared_A, out_private_A, "y_A")
output_B = classifier_head(out_shared_B, out_private_B, "y_B")
model = Model(
    inputs=[input_common_A, input_spec_A, input_common_B, input_spec_B],
    outputs=[output_A, output_B]
)
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss={"y_A": "categorical_crossentropy", "y_B": "categorical_crossentropy"},
    metrics={"y_A": ["AUC"], "y_B": ["AUC"]}
)
# ==========================================
# 3. HUẤN LUYỆN ĐỒNG THỜI (TRAINING)
# ==========================================
print("\nBắt đầu huấn luyện mô hình Shared-Private...")
history = model.fit(
    x={"X_common_A": X_common_train_A_os, "X_spec_A": X_spec_train_A_os,
       "X_common_B": X_common_train_B,    "X_spec_B": X_spec_train_B},
    y={"y_A": y_train_encoded_A_os,       "y_B": y_train_encoded_B},
    epochs=10, 
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


2026-09-21 20:42:29.979953: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)



Bắt đầu huấn luyện mô hình Shared-Private...
Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - loss: 0.8904 - y_A_AUC: 0.9294 - y_A_loss: 0.3381 - y_B_AUC: 0.7936 - y_B_loss: 0.5522 - val_loss: 0.8305 - val_y_A_AUC: 0.9617 - val_y_A_loss: 0.2896 - val_y_B_AUC: 0.8087 - val_y_B_loss: 0.5408
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.7090 - y_A_AUC: 0.9772 - y_A_loss: 0.1978 - y_B_AUC: 0.8259 - y_B_loss: 0.5112 - val_loss: 0.6497 - val_y_A_AUC: 0.9951 - val_y_A_loss: 0.1243 - val_y_B_AUC: 0.8156 - val_y_B_loss: 0.5254
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.6282 - y_A_AUC: 0.9904 - y_A_loss: 0.1306 - y_B_AUC: 0.8363 - y_B_loss: 0.4976 - val_loss: 0.5870 - val_y_A_AUC: 0.9978 - val_y_A_loss: 0.0690 - val_y_B_AUC: 0.8205 - val_y_B_loss: 0.5180
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.5751 - y_A_AUC: 0.9958 - y_A_loss: 0.0881 - y_B_AUC: 0.8430 - y_B_loss: 0.4870 - val_loss: 0.5521 - val_y_A_AUC: 0.9992 - val_y_A_loss: 0.0424

In [48]:
# ==========================================
# 4. ĐÁNH GIÁ (TESTING) TRÊN TẬP TEST GỐC CỦA BỆNH VIỆN A
# ==========================================
# Lấy đầu ra dự đoán của mô hình. 
# Ở đây ta truyền mảng 0 vào vị trí của B vì ta chỉ quan tâm dự đoán cho bệnh nhân của A.
y_pred_A = model.predict(
    {"X_common_A": X_common_test_A, "X_spec_A": X_spec_test_A,
     "X_common_B": np.zeros((len(X_common_test_A), len(common_cols))), 
     "X_spec_B": np.zeros((len(X_common_test_A), len(spec_B_cols)))}, 
    verbose=0
)
y_proba_A = y_pred_A[0][:, 1] # Lấy xác suất tử vong = 1 của nhánh A
fpr, tpr, _ = roc_curve(y_test_A, y_proba_A)
test_auc_A = auc(fpr, tpr)
print(f"\n🚀 ĐIỂM TEST AUC CỦA BỆNH VIỆN A (Shared-Private): {test_auc_A:.4f} 🚀")



🚀 ĐIỂM TEST AUC CỦA BỆNH VIỆN A (Shared-Private): 0.7461 🚀
